# Compute residual norm coefficients

This notebook computes the residual norm coefficients as part of the variable weights.

In [1]:
import os
import yaml
import copy
import numpy as np
import xarray as xr

In [2]:
from scipy.stats import gmean

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

## WRF

In [4]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [5]:
N_levels = 12

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/C404_8km/'
ds_example = xr.open_zarr(base_dir+'C404_8km_2000.zarr')
level = np.array(ds_example['bottom_top'])

In [13]:
# # get variable names
# varnames = list(conf['residual'].keys())
# varnames = varnames[:-5] # remove save_loc and others

# varname_upper = ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot', 'WRF_Q_tot_05']
# varname_surf = list(set(varnames) - set(varname_upper))

In [6]:
varname_upper = ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05']
varname_surf = ['WRF_SP', 'WRF_T2', 'WRF_TD2', 'WRF_U10', 'WRF_V10', 'WRF_PWAT_05']

In [7]:
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['residual']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['residual']['prefix'], varname)
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]
    
for varname in varname_upper:
    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['residual']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['residual']['prefix'], i_level, varname)
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std
        
    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [8]:
# separate upper air (list) and surf (float) std values
N_upper = len(varname_upper)
std_val_all = list(STD_values.values())
std_val_surf = np.array(std_val_all[:-N_upper])
std_val_upper = std_val_all[-N_upper:]

# combine
std_concat = np.concatenate([std_val_surf]+ std_val_upper)

# geometrical mean (not used)
std_g = gmean(np.sqrt(std_concat))

In [9]:
ds_std = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data) / std_g
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std[varname] = data_array
    else:
        data_array = xr.DataArray(data, name=varname)
        ds_std[varname] = data_array

In [10]:
ds_std.to_netcdf(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_residual_1980_2019_12lev.nc')

In [11]:
ds_GP = xr.open_dataset(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_residual_1980_2019_12lev_clean.nc')

ds_full = xr.open_dataset(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_residual_1980_2019_12lev.nc')

for varname in ds_full.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_full[varname].values)
    except:
        pass

=================== WRF_SP ===================
0.09474170269892271
0.09646890892812135
=================== WRF_T2 ===================
0.7590723050331437
1.038070335061416
=================== WRF_TD2 ===================
0.6361887595442969
0.9816457481303755
=================== WRF_U10 ===================
2.5823118138177117
3.644687655739152
=================== WRF_V10 ===================
2.1854107436389905
3.466621659562372
=================== WRF_PWAT_05 ===================
0.6827033851674961
1.1418341945303723
=================== WRF_P ===================
[0.09463757 0.09471565 0.09495939 0.09549486 0.09649085 0.09769533
 0.0985836  0.09898395 0.09906351 0.09953854 0.10053633 0.10540999]
[0.09642607 0.09652275 0.0967562  0.09724713 0.09822541 0.09959634
 0.10082328 0.10151499 0.10175701 0.10226155 0.10308294 0.10689739]
=================== WRF_U ===================
[2.61295577 2.17632909 1.91040001 1.77845259 1.72336151 1.51209995
 1.25372885 1.07580872 0.95809972 0.76469989 0.5727015